# Channel report: January 2024 referrals

How much of January 2024's referral traffic came from Facebook and Instagram, and how much from every other site. Reads the session export from the team's Drive folder, reduces each referrer to its site name with `tldextract`, and totals sessions into Organic Social and Referral.

In [ ]:
%pip install tldextract

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import csv

import tldextract

CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_01.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

In [ ]:
referral_rows = [row for row in rows if row["session_medium"] == "referral"]
len(referral_rows)

In [ ]:
def site_name(hostname):
    return tldextract.extract(hostname).domain.lower()

In [ ]:
channel_totals = {}
for row in referral_rows:
    if site_name(row["session_source"]) in {"facebook", "instagram"}:
        channel = "Organic Social"
    else:
        channel = "Referral"
    channel_totals[channel] = channel_totals.get(channel, 0) + int(row["sessions"])
channel_totals

## Check your work

*Added by Simmer for lesson 10.'s final exercise. Run it last, after a restart and run all.*

In [ ]:
# Check your work: run this cell last. If you change anything above, run the cells again from the top first.
# This cell only reads your variables. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import tomllib
    from pathlib import Path
    found = globals()
    results = []

    folder = Path.cwd()
    project = next((path for path in [folder, *folder.parents] if (path / "pyproject.toml").exists()), None)
    prefix = Path(sys.prefix).resolve()
    if project is None:
        results.append((False, f"This notebook isn't in a uv project's folder, but in {folder}. Move it into campaign-tools, with its data folder."))
    else:
        pyproject = tomllib.loads((project / "pyproject.toml").read_text(encoding="utf-8"))
        dev = pyproject.get("dependency-groups", {}).get("dev", [])
        if pyproject.get("project", {}).get("name") != "campaign-tools":
            results.append((False, f"This notebook is in the {project.name} project. The exercise uses campaign-tools, from lesson 9.'s final exercise: run get_files.py final in that folder."))
        elif not any(isinstance(item, str) and item.lower().startswith("jupyterlab") for item in dev):
            results.append((False, "JupyterLab isn't a development dependency of campaign-tools. In the terminal, in campaign-tools, run uv add --dev jupyterlab."))
        else:
            results.append((True, "This is the campaign-tools project, with JupyterLab as a development dependency."))

    if project is not None and prefix == (project / ".venv").resolve():
        results.append((True, f"The kernel is {project.name}'s .venv."))
    elif "archive-v0" in prefix.parts or prefix.parent.name == "tools":
        results.append((False, f"The kernel is {sys.executable}, the Python of a JupyterLab started with uvx, not the project's. Start JupyterLab in campaign-tools with uv run jupyter lab, or change the kernel to the project's."))
    else:
        results.append((False, f"The kernel is {sys.executable}, not campaign-tools's .venv. In JupyterLab, start it in campaign-tools with uv run jupyter lab. In VS Code, choose Python Environments… and the project's .venv in the kernel picker."))

    if "__vsc_ipynb_file__" in found:
        tool = "VS Code"
    elif os.environ.get("JPY_SESSION_NAME"):
        tool = "JupyterLab"
    else:
        tool = None
    if tool:
        results.append((True, f"This notebook runs in {tool}."))

    if "google.colab" in sys.modules:
        results.append((False, "google.colab is imported, so this ran in Colab. This exercise is for your own computer."))

    module = sys.modules.get("tldextract")
    if module is None:
        results.append((False, "tldextract isn't imported. Run the cell that reads the file. If it gives ModuleNotFoundError, check the kernel, not the package: it's in the project."))
    elif project is not None and not Path(module.__file__).resolve().is_relative_to((project / ".venv").resolve()):
        results.append((False, f"tldextract imports from {Path(module.__file__).parent}, not from the project's .venv. Change the kernel to the project's .venv, then restart and run all."))
    else:
        results.append((True, "tldextract imports from the project's .venv."))

    path = found.get("CSV_PATH")
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Run the cell that reads the file."))
    elif path.startswith("/content"):
        results.append((False, f"CSV_PATH is still Colab's path, {path!r}. The file is in the data folder next to this notebook: \"data/sessions_export_2024_01.csv\"."))
    elif Path(path).is_absolute():
        results.append((False, f"CSV_PATH is {path!r}, which only works on your computer. Use the path relative to this notebook: \"data/sessions_export_2024_01.csv\"."))
    elif not path.replace("\\", "/").endswith("sessions_export_2024_01.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The report uses January's export, sessions_export_2024_01.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}, relative to {folder}. Check that the data folder is next to this notebook. If it isn't there, run get_files.py final again."))
    else:
        results.append((True, "CSV_PATH is a relative path to January's export."))

    rows = found.get("rows")
    referral_rows = found.get("referral_rows")
    totals = found.get("channel_totals")
    if not isinstance(rows, list) or len(rows) != 1711:
        results.append((False, "rows should have the 1,711 rows of January's export. Run the cell that reads the file."))
    else:
        results.append((True, "rows has all 1,711 rows of the export."))
    if not isinstance(referral_rows, list) or len(referral_rows) != 365:
        results.append((False, "referral_rows should have the 365 referral rows. Don't change the cells that calculate the report: run them again from the top."))
    else:
        results.append((True, "referral_rows has the 365 referral rows."))
    if totals == {"Organic Social": 4640, "Referral": 2655}:
        results.append((True, "channel_totals has the right sessions for both channels."))
    else:
        results.append((False, f"channel_totals should be {{'Referral': 2655, 'Organic Social': 4640}}, but it is {totals!r}. Don't change the cells that calculate the report: restart and run all."))

    for passed, message in results:
        print(("✅ " if passed else "❌ ") + message)
    if not all(passed for passed, message in results):
        print("\nFix the first ❌, restart the kernel and run all, then check the result again.")
    elif tool == "JupyterLab":
        print("\nEverything matches in JupyterLab. Now open the notebook in VS Code, choose the project's .venv, and run all again.")
    elif tool == "VS Code":
        print("\nEverything matches in VS Code. If it matched in JupyterLab too, both tools give the same result with the same kernel.")
    else:
        print("\nEverything matches. Run the notebook in JupyterLab and in VS Code, and check that you get the same result in both.")


check_your_work()